# Soil 16S Phylogeny Pilot

## Species finding and relatedness from a class-safe cache

Run the cells from top to bottom. When you see a **Think** prompt, pause and write one sentence.

Today you will:

1. load cached 16S rRNA reference sequences from soil-relevant bacteria,
2. compare two unknown soil ASV/read sequences with those references,
3. inspect a small aligned marker window,
4. turn sequence differences into a distance matrix,
5. build UPGMA and neighbor-joining trees,
6. report a careful closest-reference claim.

A tree is a hypothesis from evidence. In this notebook the evidence is one short 16S marker window, so the final claim must stay cautious.

This is a browser-only 16S marker/metabarcoding pilot. It is not a shotgun metagenomics pipeline and it does not prove exact species identity.


In [ ]:
#@title Class controls { display-mode: "form" }
USE_GITHUB_CACHE = False #@param {type:"boolean"}
CACHE_BASE_URL = "" #@param {type:"string"}
QUERY_TO_REPORT = "Soil_ASV_A" #@param ["Soil_ASV_A", "Soil_ASV_B"]
TREE_METHOD_TO_SHOW = "Compare UPGMA and neighbor joining" #@param ["UPGMA", "Neighbor joining", "Compare UPGMA and neighbor joining"]
MARKER_WINDOW_BASES = 520 #@param {type:"slider", min:200, max:560, step:20}
ALIGNMENT_START = 130 #@param {type:"slider", min:0, max:420, step:10}
ALIGNMENT_WIDTH = 70 #@param {type:"slider", min:30, max:100, step:10}
print("Controls set. The default path uses the embedded cache, so class runs do not depend on live BLAST.")


In [ ]:
#@title Install and import notebook dependencies { display-mode: "form" }
import importlib.util
import subprocess
import sys

def ensure(package, import_name=None):
    import_name = import_name or package
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

for package, import_name in [
    ("biopython", "Bio"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
]:
    ensure(package, import_name)

import json
import math
import urllib.request
import xml.etree.ElementTree as ET
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML, display

from Bio import Phylo, SeqIO
from Bio.Align import PairwiseAligner
from Bio.Phylo.TreeConstruction import DistanceMatrix, DistanceTreeConstructor

EMBEDDED_CACHE = {
  "pilot_16s_references.fasta": ">Bacillus_subtilis_168 accession=NR_102783.2 source=NCBI_Nucleotide role=reference species=\"Bacillus subtilis subsp. subtilis strain 168\"\nTTATCGGAGAGTTTGATCCTGGCTCAGGACGAACGCTGGCGGCGTGCCTAATACATGCAAGTCGAGCGGACAGATGGGAG\nCTTGCTCCCTGATGTTAGCGGCGGACGGGTGAGTAACACGTGGGTAACCTGCCTGTAAGACTGGGATAACTCCGGGAAAC\nCGGGGCTAATACCGGATGGTTGTTTGAACCGCATGGTTCAAACATAAAAGGTGGCTTCGGCTACCACTTACAGATGGACC\nCGCGGCGCATTAGCTAGTTGGTGAGGTAACGGCTCACCAAGGCGACGATGCGTAGCCGACCTGAGAGGGTGATCGGCCAC\nACTGGGACTGAGACACGGCCCAGACTCCTACGGGAGGCAGCAGTAGGGAATCTTCCGCAATGGACGAAAGTCTGACGGAG\nCAACGCCGCGTGAGTGATGAAGGTTTTCGGATCGTAAAGCTCTGTTGTTAGGGAAGAACAAGTGCCGTTCGAATAGGGCG\nGTACCTTGACGGTACCTAACCAGAAAGCCACGGCTAACTACGTGCCAGCAGCCGCGGTAATACGTAGGTGGCAAGCGTTG\nTCCGGAATTATTGGGCGTAAAGGGCTCGCAGGCGGTTTCTTAAGTCTGATGTGAAAGCCCCCGGCTCAACCGGGGAGGGT\nCATTGGAAACTGGGGAACTTGAGTGCAGAAGAGGAGAGTGGAATTCCACGTGTAGCGGTGAAATGCGTAGAGATGTGGAG\nGAACACCAGTGGCGAAGGCGACTCTCTGGTCTGTAACTGACGCTGAGGAGCGAAAGCGTGGGGAGCGAACAGGATTAGAT\nACCCTGGTAGTCCACGCCGTAAACGATGAGTGCTAAGTGTTAGGGGGTTTCCGCCCCTTAGTGCTGCAGCTAACGCATTA\nAGCACTCCGCCTGGGGAGTACGGTCGCAAGACTGAAACTCAAAGGAATTGACGGGGGCCCGCACAAGCGGTGGAGCATGT\nGGTTTAATTCGAAGCAACGCGAAGAACCTTACCAGGTCTTGACATCCTCTGACAATCCTAGAGATAGGACGTCCCCTTCG\nGGGGCAGAGTGACAGGTGGTGCATGGTTGTCGTCAGCTCGTGTCGTGAGATGTTGGGTTAAGTCCCGCAACGAGCGCAAC\nCCTTGATCTTAGTTGCCAGCATTCAGTTGGGCACTCTAAGGTGACTGCCGGTGACAAACCGGAGGAAGGTGGGGATGACG\nTCAAATCATCATGCCCCTTATGACCTGGGCTACACACGTGCTACAATGGACAGAACAAAGGGCAGCGAAACCGCGAGGTT\nAAGCCAATCCCACAAATCTGTTCTCAGTTCGGATCGCAGTCTGCAACTCGACTGCGTGAAGCTGGAATCGCTAGTAATCG\nCGGATCAGCATGCCGCGGTGAATACGTTCCCGGGCCTTGTACACACCGCCCGTCACACCACGAGAGTTTGTAACACCCGA\nAGTCGGTGAGGTAACCTTTTAGGAGCCAGCCGCCGAAGGTGGGACAGATGATTGGGGTGAAGTCGTAACAAGGTAGCCGT\nATCGGAAGGTGCGGCTGGATCACCTCCTTT\n>Pseudomonas_fluorescens_CCM2115 accession=NR_115715.1 source=NCBI_Nucleotide role=reference species=\"Pseudomonas fluorescens strain CCM 2115\"\nAGAGTTTGATCCTGGCTCAGATTGAACGCTGGCGGCAGGCCTAACACATGCAAGTCGAGCGGTAGAGAGAAGCTTGCTTC\nTCTTGAGAGCGGCGGACGGGTGAGTAAAGCCTAGGAATCTGCCTGGTAGTGGGGGATAACGTTCGGAAACGGACGCTAAT\nACCGCATACGTCCTACGGGAGAAAGCAGGGGACCTTCGGGCCTTGCGCTATCAGATGAGCCTAGGTCGGATTAGCTAGTT\nGGTGAGGTAATGGCTCACCAAGGCGACGATCCGTAACTGGTCTGAGAGGATGATCAGTCACACTGGAACTGAGACACGGT\nCCAGACTCCTACGGGAGGCAGCAGTGGGGAATATTGGACAATGGGCGAAAGCCTGATCCAGCCATGCCGCGTGTGTGAAG\nAAGGTCTTCGGATTGTAAAGCACTTTAAGTTGGGAGGAAGGGCATTAACCTAATACGTTAGTGTTTTGACGTTACCGACA\nGAATAAGCACCGGCTAACTCTGTGCCAGCAGCCGCGGTAATACAGAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAA\nAGCGCGCGTAGGTGGTTTGTTAAGTTGGATGTGAAATCCCCGGGCTCAACCTGGGAACTGCATTCAAAACTGACTGACTA\nGAGTATGGTAGAGGGTGGTGGAATTTCCTGTGTAGCGGTGAAATGCGTAGATATAGGAAGGAACACCAGTGGCGAAGGCG\nACCACCTGGACTAATACTGACACTGAGGTGCGAAAGCGTGGGGAGCAAACAGGATTAGATACCCTGGTAGTCCACGCCGT\nAAACGATGTCAACTAGCCGTTGGGAGCCTTGAGCTCTTAGTGGCGCAGCTAACGCATTAAGTTGACCGCCTGGGGAGTAC\nGGCCGCAAGGTTAAAACTCAAATGAATTGACGGGGGCCCGCACAAGCGGTGGAGCATGTGGTTTAATTCGAAGCAACGCG\nAAGAACCTTACCAGGCCTTGACATCCAATGAACTTTCTAGAGATAGATTGGTGCCTTCGGGAACATTGAGACAGGTGCTG\nCATGGCTGTCGTCAGCTCGTGTCGTGAGATGTTGGGTTAAGTCCCGTAACGAGCGCAACCCTTGTCCTTAGTTACCAGCA\nCGTAATGGTGGGCACTCTAAGGAGACTGCCGGTGACAAACCGGAGGAAGGTGGGGATGACGTCAAGTCATCATGGCCCTT\nACGGCCTGGGCTACACACGTGCTACAATGGTCGGTACAGAGGGTTGCCAAGCCGCGAGGTGGAGCTAATCCCACAAAACC\nGATCGTAGTCCGGATCGCAGTCTGCAACTCGACTGCGTGAAGTCGGAATCGCTAGTAATCGCGAATCAGAATGTCGCGGT\nGAATACGTTCCCGGGCCTTGTACACACCGCCCGTCACACCATGGGAGTGGGTTGCACCAGAAGTAGCTAGTCTAACCTTC\nGGGAGGACGGTTACCACGGTGTGATTCATGACTGGGGTGAAGTCGTAACAAGGTAGCCGTAGGGGAACCTGCGGCTGGAT\n>Streptomyces_coelicolor_rrnD accession=Y00411.1 source=NCBI_Nucleotide role=reference species=\"Streptomyces coelicolor\"\nTGGGCCCGCATCACCATCGGCGTCCTCGCCGAGCTGGCCTTCCTGGCCTACGTCTACGTTCTGGGCGGCCGAGCCGTGCG\nCGACGGCGAGACGGGTGACGTCGAGGCAGCCGAACGCAGCGCCACGGTGCCAACAGCCGCCTGATGTGCATCCACCCCTG\nCGAGCTGCTAGTGTCCTCTTCGTTCCCGCAAGAGCCGTTGACACGGAGCGAGCGGGGAGGTAGATTCGAACAGTTGCCTG\nGAGACGGGTTCACCCCAGAGGGCAACAGTGAACATCTACCAGCTTCTCCGAATCAACGAATTCGACGAAGCACTCTCCCG\nATGAATCGGAAACGAAGGCCGGTAAGACCGGCTCGAAAGTTCTGATAAAGTCGGAGCCGCCGGAAAGGGAAACGCGAAAG\nCGGGAACCTGGAAAGCGCCGAGGAAATCGGATCGGAAAGATCTGATAGAGTCGGAAACGCAAGACCGAAGGGAAGCGCCC\nGGAGGAAAGCCCGAGAGGGTGAGTACAAAGGAAGCGTCCGTTCCTTGAGAACTCAACAGCGTGCCAAAAGTCAACGCCAG\nATATGTTGATACCCCGACCTGATCGGATCTCCGTTCGGGTTGAGGTTCCTTTGAAGTAACACAACAGCGAGGACGCTGTG\nAACGGTCGGATTATTCCTCCGACTGTTCCGCTCTCGTGGTGTCACCCGATTACGGGTATACATTCACGGAGAGTTTGATC\nCTGGCTCAGGACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGATGAACCACTTCGGTGGGGATTAGTGGCGA\nACGGGTGAGTAACACGTGGGCAATCTGCCCTTCACTCTGGGACAAGCCCTGGAAACGGGGTCTAATACCGGATACTGACC\nCTCGCAGGCATCTGCGAGGTTCGAAAGCTCCGGCGGTGAAGGATGAGCCCGCGGCCTATCAGCTTGTTGGTGAGGTAATG\nGCTCACCAAGGCGACGACGGGTAGCCGGCCTGAGAGGGCGACCGGCCACACTGGGACTGAGACACGGCCCAGACTCCTAC\nGGGAGGCAGCAGTGGGGAATGTTGCACAATGGGCGAAAGCCTGATGCAGCGACGCCGCGTGAGGGATGACGGCCTTCGGG\nTTGTAAACCTCTTTCAGCAGGGAAGAAGCGAAAGTGACGGTACCTGCAGAAGAAGCGCCGGCTAACTACGTGCCAGCAGC\nCGCGGTAATACGTAGGGCGCAAGCGTTGTCCGGAATTATTGGGCGTAAAGAGCTCGTAGGCGGCTTGTCACGTCGGTTGT\nGAAAGCCCGGGGCTTAACCCCGCCACTGCAGTCGATACGGGCAGGCTAGAGTTCGGTAGGGGAGATCGGAATTCCTGGTG\nTAGCGGTGAAATGCGCAGATATCAGGAGGAACACCGGTGGCGAAGGCGGATCTCTGGGCCGATACTGACGCTGAGGAGCG\nAAAGNGTGGGGAGCGAACAGGATTAGATACCCTGGTAGTCCACGCCGTAAACGGTGGGCACTAGGTGTGGGCAACATTCC\nACGTTGTCCGTGCCGCAGCTAACGCATTAAGTGCCCCGCCTGGGGAGTACGGCCGCAAGGCTAAAACTCAAAGGAATTGA\nCGGGGGCCCGCACAAGCGGCGGAGCATGTGGCTTAATTCGACGCAACGCGAAGAACCTTACCAAGGCTTGACATACACCG\nGAAAGCATCAGAGATGGTGCCCCCCTTGTGGTCGGTGTACAGGTGGTGCATGGCTGTCGTCAGCTCGTGTCGTGAGATGT\nTGGGTTAAGTCCCGCAACGAGCGCAACCCTTGTCCCGTGTTGCCAGCAAGCCCTTCGGGGTGTTGGGGACTCACGGGAGA\nCCGCCGGGGTCAACTCGGAGGAAGGTGGGGACGACGTCAAGTCATCATGCCCCTTATGTCTTGGGCTGCACACGTGCTAC\nAATGGCCGGTACAATGAGCTGCGATACCGCAAGGTGGAGCGAATCTCAAAAAGCCGGTCTCAGTTCGGATTGGGGTCTGC\nAACTCGACCCCATGAAGTCGGAGTCGCTAGTAATCGCAGATCAGCATTGCTGCGGTGAATACGTTCCCGGGCCTTGTACA\nCACCGCCCGTCACGTCACGAAAGTCGGTAACACCCGAAGCCGGTGGCCCAACCCCTTGTGGGAGGGAGCTGTCGAAGGTG\nGGACTGGCGATTGGGACGAAGTCGTAACAAGGTAGCCGTACCGGAAGGTGCGGCTGGATCACCTCCTTTCTAAGGAGCAC\nATAGCCGACTGCAGCGAAATGTCCTGCACGGTTGCTCATGGGTGGAACGTTGACTACTCGGCACGGTCTTCTTGATGGAT\nCACTAGTACTGCTTCGGCGTGGAACGTGACTTCAAAGAGGGGTTCGTGTCGGGCACGCTGTTGGGTATCTGAGGGTACGG\nCCGTGAGGTCGCCTTCAGTTGCCGGCCCCGGTAAAAATCCGCGTGAGTGGGTTGTGACGGGTGGTTGGTCGTTGTTTGAG\nAACTGCACAGTGGACGCGAGCATCTGTGGCCAAGTTTTTAAGGGCGCACGGTGGATGCCTT\n>Rhizobium_leguminosarum_IAM12609 accession=D14513.1 source=NCBI_Nucleotide role=reference species=\"Rhizobium leguminosarum type strain IAM 12609\"\nAACTTGAGAGTTTGATCCTGGCTCAGAACGAACGCTGGCGGCAGGCTTAACACATGCAAGTCGAGCGCCCCGCAANNNNA\nGCGGCAGACGGGTGAGTAACGCGTGGGAACGTACCCTTTACTACGGAATAACGCAGGGAAACTTGTGCTAATACCGTATG\nTGCCCTTTGGGGGAAAGATTTATCGGTAAAGGATCGGCCCGCGTTGGATTAGCTAGTTGGTGGGGTAAAGGCCTACCAAG\nGCGACGATCCATAGCTGGTCTGAGAGGATGATCAGCCACATTGGGACTGAGACACGGCCCAAACTCCTACGGGAGGCAGC\nAGTGGGGAATATTGGACAATGGGCGCAAGCCTGATCCAGCCATGCCGCGTGAGTGATGAAGGCCCTAGGGTTGTAAAGCT\nCTTTCACCGGAGAAGATAATGACGGTATCCGGAGAAGAAGCCCCGGCTAACTTCGTGCCAGCAGCCGCGGTAATACGAAG\nGGGGCTAGCGTTGTTCGGAATTACTGGGCGTAAAGCGCACGTAGGCGGATCGATAAGTCAGGGGTGAAATCCCAGGGCTC\nAACCCTGGAACTGCCTTTGATACTGTCGATCTGGAGTATGGAAGAGGTGAGTGGAATTCCGAGTGTAGAGGTGAAATTCG\nTAGATATTCGGAGGAACACCAGTGGCGAAGGCGGCTCACTGGTCCATTACTGACGCTGAGGTGCGAAAGCGTGGGGAGCA\nAACAGGATTAGATACCCTGGTAGTCCACGCCGTAAACGATGAATGTTAGCCGTCGGGCAGTATACTGTTCGGTGGCGCAC\nGTAACGCATTAAACATTCCGCCTGGGGAGTACGGTCGCAAGATTAAAACTCAAAGGAATTGACGGGGGCCCGCACAAGCG\nGTGGAGCATGTGGTTTAATTCGAAGCAACGCGCAGAACCTTACCAGCCCTTGACATGCCCGGCTACTTGCAGAGATGCAA\nGGTTCTTCGGGGACCGGGACACAGGTGCTGCATGGCTGTCGTCAGCTCGTGTCGTGAGATGTTGGGTTAAGTCCCGCAAC\nGAGCGCAACCCTCGCCCTTAGTTGCCAGCATTCAGTTGGGCACTCTAAGGGGACTGCCGGTGATAAGCCGAGAGGAAGGT\nGGGGATGACGTCAAGTCCTCATGGCCCTTACGGGCTGGGCTACACACGTGCTACAATGGTGGTGACAGTGGGCAGCGAGC\nACGCGAGTGTGAGCTAATCTCCAAAAGCCATCTCAGTTCGGATTGCACTCTGCAACTCGAGTGCATGAAGTTGGAATCGC\nTAGTAATCGCGGATCAGCATGCCGCGGTGAATACGTTCCCGGGCCTTGTACACACCGCCCGTCACACCATGGGAGTTGGT\nTTTACCCGAAGGTAGTGCGCTAACCGCAAGGAGGCAGCTAACCACGGTAGGGTCAGCGACTGGGGTGAAGTCGTAACAAG\nGTAGCCGTAGGGGAACCTGCGGCTGGATCACCTCC\n>Acidobacterium_capsulatum_ATCC51196 accession=NR_074106.1 source=NCBI_Nucleotide role=reference species=\"Acidobacterium capsulatum ATCC 51196\"\nAGAGTTTGATCCTGGCTCAGAATCAACGCTGGCGGCGTGCCTAACACATGCAAGTCGAACAAGAAAGGGACTTCGGTCCT\nGAGTACAGTGGCGCACGGGTGAGTAACACGTGACTAACCTACCCTCGAGTGGGGAATAACTTCGGGAAACCGAGGCTAAT\nACCGCATAATACCCACGGGTCAAAGGAGCAATTCGCTTGAGGAGGGGGTCGCGGCCGATTAGCTAGTTGGCGGGGTAATG\nGCCCACCAAGGCAGTGATCGGTATCCGGCCTGAGAGGGCGCACGGACACACTGGAACTGAAACACGGTCCAGACTCCTAC\nGGGAGGCAGCAGTGGGGAATTTTGCGCAATGGGGGAAACCCTGACGCAGCAACGCCGCGTGGAGGATGAAGTCTCTTGGG\nACGTAAACTCCTTTCGATCGGAACGATTATGACGGTACCGGAAGAAGAAGCCCCGGCTAACTTCGTGCCAGCAGCCGCGG\nTAATACGAGGGGGGCGAGCGTTGTTCGGAATTATTGGGCGTAAAGGGTGCGTAGGCGGTTCGGTAAGTTTGATGTGAAAT\nCTTCGGGCTCAACTCGAAGTCTGCATCGAAAACTGCCGGGCTTGAGTGTGGGAGAGGTGAGTGGAATTTCCGGTGTAGCG\nGTGAAATGCGTAGATATCGGAAGGAACACCTGTGGCGAAAGCGGCTCACTGGACCACAACTGACGCTGATGCACGAAAGC\nTAGGGGAGCAAACAGGATTAGATACCCTGGTAGTCCTAGCCCTAAACGATGATCGCTTGGTGTGGCGGGTACCCAATCCC\nGTCGTGCCGTAGCTAACGCGTTAAGCGATCCGCCTGGGGAGTACGGTCGCAAGGCTGAAACTCAAAGGAATTGACGGGGG\nCCCGCACAAGCGGTGGAGCATGTGGTTTAATTCGACGCAACGCGAAGAACCTTACCTGGGCTCGAAATGTAGTGGACCGG\nGGTAGAAATATCCCTTCCCCGCAAGGGGCTGCTATATAGGTGCTGCATGGCTGTCGTCAGCTCGTGTCGTGAGATGTTGG\nGTTAAGTCCCGCAACGAGCGCAACCCTTATTGCCAGTTGCTACCATTTAGTTGAGCACTCTGGTGAGACCGCCTCGGATA\nACGGGGAGGAAGGTGGGGATGACGTCAAGTCCTCATGGCCTTTATGTCCAGGGCTACACACGTGCTACAATGGCCGGTAC\nAAACCGCCGCAAACCCGCGAGGGGGAGCTAATCGGAAAAAGCCGGCCTCAGTTCGGATTGTAGTCTGCAACTCGACTACA\nTGAAGCTGGAATCGCTAGTAATCGCGGATCAGCATGCCGCGGTGAATACGTTCCCGGGCCTTGTACACACCGCCCGTCAC\nATCACGAAAGTGGGTCGTACTAGAAGCGGGTGAGCCAACCGTAAGGAGGCAGCCTTCCAAGGTGTGATTCATGATTGGGG\nTGAAGTCGTAACAAGGTAGCCGTAGGAGAACCTGCGGCTGGATCACCTCCTTT\n",
  "pilot_16s_query_reads.fasta": ">Soil_ASV_A source_accession=NR_102783.2 source=teaching_cache role=query\nAGAGTTTGATCCTGGCTCAGGACGAACGCTGGCGGCGTGCCTACTACATGCAAGTCGAGCGGACAGATGGGAGCTTGCTC\nCCTGATGTTAGCGGCGGACGGGTGAGTAACACGTGGGTAACCTGCCTGTAAGACTGGGATACCTCCGGGAAACCGGGGCT\nAATACCGGATGGTTGTTTGAACCGCATGGTTCAAACATAAAAGGTGGCTTCGGCTACCACTTACAGATGGACCCGCGGCG\nCATTAGCTAGTTGGTGAGGTAACGGCTCACCAAGGCGACGATGCGTAGCCGACCTGAGAGGGTGATCGGGCACACTGGGA\nCTGAGACACGGCCCAGACTCCTACGGGAGGCAGCAGTAGGGAATCTTCCGCAATGGACGAAAGTCTGACGGAGCAACGCC\nGCGTGAGTGATGAAGGTTTTCGGATCGTAAAGCTCTGTTGTTAGGGAAGAACAAGTGCCGTTCGAATAGGGGGGTACCTT\nGACGGTACCTAACCAGAAAGCCACGGCTAACTACGTGCCA\n>Soil_ASV_B source_accession=D14513.1 source=teaching_cache role=query\nAGAGTTTGATCCTGGCTCAGAACGAACGCTGGCGGCAGGCTTAACACATGCAAGTCGATCGCCCCGCAANNNNAGCGGCA\nGACGGGTGAGTAACGCGTGGGAACGTACCCTTTACTACGGAATAACGCAGGGAAACTTGTGCTAATACCGTATGTGCCCT\nTTGGGGGAAAGATTTATCGGTAAAGGATCGGCCCGCGTTGGATTAGCTAGTTGGTGGGGTCAAGGCCTACCAAGGCGACG\nATCCATAGCTGGTCTGAGAGGATGATCAGCCACATTGGGACTGAGACACGGCCCAAACTCCTACGGGAGGCAGCAGTGGG\nGAATATTGGACAATGGGCGCAAGCCTGATCCAGCCATGCCGCGTGAGTGATGAAGGCCCTAGGGTTGTAAATCTCTTTCA\nCCGGAGAAGATAATGACGGTATCCGGAGAAGAAGCCCCGGCTAACTTCGTGCCAGCAGCCGCGGTAATACGAAGGGGGCT\nAGCGTTGTTCGGAATTACTGGGCGTCAAGCGCACGTAGGC\n",
  "pilot_16s_metadata.csv": "label,role,species_or_query,accession,source_database,source_url,retrieved_date,phylum,color,soil_context,note\nBacillus_subtilis_168,reference,Bacillus subtilis subsp. subtilis strain 168,NR_102783.2,NCBI Nucleotide,https://www.ncbi.nlm.nih.gov/nuccore/NR_102783.2,2026-05-25,Bacillota,#E69F00,common soil and rhizosphere model bacterium,\"Gram-positive, spore-forming soil bacterium; useful classroom decomposer reference.\"\nPseudomonas_fluorescens_CCM2115,reference,Pseudomonas fluorescens strain CCM 2115,NR_115715.1,NCBI Nucleotide,https://www.ncbi.nlm.nih.gov/nuccore/NR_115715.1,2026-05-25,Pseudomonadota,#56B4E9,rhizosphere-associated soil bacterium,Common plant-root associated reference; useful contrast to Gram-positive taxa.\nStreptomyces_coelicolor_rrnD,reference,Streptomyces coelicolor,Y00411.1,NCBI Nucleotide,https://www.ncbi.nlm.nih.gov/nuccore/Y00411.1,2026-05-25,Actinomycetota,#009E73,filamentous soil actinomycete,Classic soil actinomycete; illustrates that soil microbes are not a single close group.\nRhizobium_leguminosarum_IAM12609,reference,Rhizobium leguminosarum type strain IAM 12609,D14513.1,NCBI Nucleotide,https://www.ncbi.nlm.nih.gov/nuccore/D14513.1,2026-05-25,Pseudomonadota,#56B4E9,root nodule and nitrogen-cycling context,Plant-associated nitrogen-cycle reference; sequence contains a few ambiguous N bases from the original record.\nAcidobacterium_capsulatum_ATCC51196,reference,Acidobacterium capsulatum ATCC 51196,NR_074106.1,NCBI Nucleotide,https://www.ncbi.nlm.nih.gov/nuccore/NR_074106.1,2026-05-25,Acidobacteriota,#CC79A7,acidic soil and broad soil ecology reference,Soil-relevant reference from a major soil-associated phylum.\nSoil_ASV_A,query,unknown soil ASV A,NR_102783.2,Teaching cache derived from NCBI reference,https://www.ncbi.nlm.nih.gov/nuccore/NR_102783.2,2026-05-25,Teaching query,#222222,class teaching query from a rhizosphere-style sample,Synthetic classroom read derived from the Bacillus subtilis 16S marker window with a few substitutions.\nSoil_ASV_B,query,unknown soil ASV B,D14513.1,Teaching cache derived from NCBI reference,https://www.ncbi.nlm.nih.gov/nuccore/D14513.1,2026-05-25,Teaching query,#222222,class teaching query from a root-associated soil sample,Synthetic classroom read derived from the Rhizobium leguminosarum 16S marker window with a few substitutions.\n",
  "pilot_16s_cached_hits.csv": "query_label,rank,reference_label,reference_accession,reference_species,compared_bases,differences,fraction_different,percent_identity_teaching_window,source,interpretation\nSoil_ASV_A,1,Bacillus_subtilis_168,NR_102783.2,Bacillus subtilis subsp. subtilis strain 168,520,4,0.007692,99.231,precomputed class-safe teaching hit table,closest cached reference\nSoil_ASV_A,2,Streptomyces_coelicolor_rrnD,Y00411.1,Streptomyces coelicolor,483,96,0.198758,80.124,precomputed class-safe teaching hit table,lower-ranked cached reference\nSoil_ASV_A,3,Rhizobium_leguminosarum_IAM12609,D14513.1,Rhizobium leguminosarum type strain IAM 12609,450,98,0.217778,78.222,precomputed class-safe teaching hit table,lower-ranked cached reference\nSoil_ASV_A,4,Acidobacterium_capsulatum_ATCC51196,NR_074106.1,Acidobacterium capsulatum ATCC 51196,470,106,0.225532,77.447,precomputed class-safe teaching hit table,lower-ranked cached reference\nSoil_ASV_A,5,Pseudomonas_fluorescens_CCM2115,NR_115715.1,Pseudomonas fluorescens strain CCM 2115,503,119,0.236581,76.342,precomputed class-safe teaching hit table,lower-ranked cached reference\nSoil_ASV_B,1,Rhizobium_leguminosarum_IAM12609,D14513.1,Rhizobium leguminosarum type strain IAM 12609,514,3,0.005837,99.416,precomputed class-safe teaching hit table,closest cached reference\nSoil_ASV_B,2,Pseudomonas_fluorescens_CCM2115,NR_115715.1,Pseudomonas fluorescens strain CCM 2115,480,90,0.187500,81.250,precomputed class-safe teaching hit table,lower-ranked cached reference\nSoil_ASV_B,3,Streptomyces_coelicolor_rrnD,Y00411.1,Streptomyces coelicolor,513,102,0.198830,80.117,precomputed class-safe teaching hit table,lower-ranked cached reference\nSoil_ASV_B,4,Bacillus_subtilis_168,NR_102783.2,Bacillus subtilis subsp. subtilis strain 168,490,102,0.208163,79.184,precomputed class-safe teaching hit table,lower-ranked cached reference\nSoil_ASV_B,5,Acidobacterium_capsulatum_ATCC51196,NR_074106.1,Acidobacterium capsulatum ATCC 51196,506,112,0.221344,77.866,precomputed class-safe teaching hit table,lower-ranked cached reference\n",
  "pilot_16s_cached_blast.xml": "<?xml version=\"1.0\"?>\n<BlastOutput>\n  <BlastOutput_program>blastn</BlastOutput_program>\n  <BlastOutput_version>cached-teaching-blast-1.0</BlastOutput_version>\n  <BlastOutput_reference>Class-safe teaching XML generated from precomputed aligned-window hits; not a live NCBI BLAST run.</BlastOutput_reference>\n  <BlastOutput_db>soil_16s_class_cache</BlastOutput_db>\n  <BlastOutput_query-ID>soil_16s_teaching_queries</BlastOutput_query-ID>\n  <BlastOutput_query-def>soil_16s_teaching_queries</BlastOutput_query-def>\n  <BlastOutput_param><Parameters><Parameters_matrix>identity</Parameters_matrix></Parameters></BlastOutput_param>\n  <BlastOutput_iterations>\n    <Iteration>\n      <Iteration_iter-num>1</Iteration_iter-num>\n      <Iteration_query-ID>Soil_ASV_A</Iteration_query-ID>\n      <Iteration_query-def>Soil_ASV_A</Iteration_query-def>\n      <Iteration_query-len>520</Iteration_query-len>\n      <Iteration_hits>\n        <Hit>\n          <Hit_num>1</Hit_num>\n          <Hit_id>Bacillus_subtilis_168</Hit_id>\n          <Hit_def>Bacillus subtilis subsp. subtilis strain 168</Hit_def>\n          <Hit_accession>NR_102783.2</Hit_accession>\n          <Hit_len>520</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>1028</Hsp_bit-score>\n              <Hsp_evalue>1e-120</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>520</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>520</Hsp_hit-to>\n              <Hsp_identity>516</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>520</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n        <Hit>\n          <Hit_num>2</Hit_num>\n          <Hit_id>Streptomyces_coelicolor_rrnD</Hit_id>\n          <Hit_def>Streptomyces coelicolor</Hit_def>\n          <Hit_accession>Y00411.1</Hit_accession>\n          <Hit_len>483</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>678</Hsp_bit-score>\n              <Hsp_evalue>1e-80</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>483</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>483</Hsp_hit-to>\n              <Hsp_identity>387</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>483</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n        <Hit>\n          <Hit_num>3</Hit_num>\n          <Hit_id>Rhizobium_leguminosarum_IAM12609</Hit_id>\n          <Hit_def>Rhizobium leguminosarum type strain IAM 12609</Hit_def>\n          <Hit_accession>D14513.1</Hit_accession>\n          <Hit_len>450</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>606</Hsp_bit-score>\n              <Hsp_evalue>1e-75</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>450</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>450</Hsp_hit-to>\n              <Hsp_identity>352</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>450</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n        <Hit>\n          <Hit_num>4</Hit_num>\n          <Hit_id>Acidobacterium_capsulatum_ATCC51196</Hit_id>\n          <Hit_def>Acidobacterium capsulatum ATCC 51196</Hit_def>\n          <Hit_accession>NR_074106.1</Hit_accession>\n          <Hit_len>470</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>622</Hsp_bit-score>\n              <Hsp_evalue>1e-70</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>470</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>470</Hsp_hit-to>\n              <Hsp_identity>364</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>470</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n        <Hit>\n          <Hit_num>5</Hit_num>\n          <Hit_id>Pseudomonas_fluorescens_CCM2115</Hit_id>\n          <Hit_def>Pseudomonas fluorescens strain CCM 2115</Hit_def>\n          <Hit_accession>NR_115715.1</Hit_accession>\n          <Hit_len>503</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>649</Hsp_bit-score>\n              <Hsp_evalue>1e-65</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>503</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>503</Hsp_hit-to>\n              <Hsp_identity>384</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>503</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n      </Iteration_hits>\n    </Iteration>\n    <Iteration>\n      <Iteration_iter-num>2</Iteration_iter-num>\n      <Iteration_query-ID>Soil_ASV_B</Iteration_query-ID>\n      <Iteration_query-def>Soil_ASV_B</Iteration_query-def>\n      <Iteration_query-len>514</Iteration_query-len>\n      <Iteration_hits>\n        <Hit>\n          <Hit_num>1</Hit_num>\n          <Hit_id>Rhizobium_leguminosarum_IAM12609</Hit_id>\n          <Hit_def>Rhizobium leguminosarum type strain IAM 12609</Hit_def>\n          <Hit_accession>D14513.1</Hit_accession>\n          <Hit_len>514</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>1019</Hsp_bit-score>\n              <Hsp_evalue>1e-120</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>514</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>514</Hsp_hit-to>\n              <Hsp_identity>511</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>514</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n        <Hit>\n          <Hit_num>2</Hit_num>\n          <Hit_id>Pseudomonas_fluorescens_CCM2115</Hit_id>\n          <Hit_def>Pseudomonas fluorescens strain CCM 2115</Hit_def>\n          <Hit_accession>NR_115715.1</Hit_accession>\n          <Hit_len>480</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>690</Hsp_bit-score>\n              <Hsp_evalue>1e-80</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>480</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>480</Hsp_hit-to>\n              <Hsp_identity>390</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>480</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n        <Hit>\n          <Hit_num>3</Hit_num>\n          <Hit_id>Streptomyces_coelicolor_rrnD</Hit_id>\n          <Hit_def>Streptomyces coelicolor</Hit_def>\n          <Hit_accession>Y00411.1</Hit_accession>\n          <Hit_len>513</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>720</Hsp_bit-score>\n              <Hsp_evalue>1e-75</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>513</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>513</Hsp_hit-to>\n              <Hsp_identity>411</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>513</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n        <Hit>\n          <Hit_num>4</Hit_num>\n          <Hit_id>Bacillus_subtilis_168</Hit_id>\n          <Hit_def>Bacillus subtilis subsp. subtilis strain 168</Hit_def>\n          <Hit_accession>NR_102783.2</Hit_accession>\n          <Hit_len>490</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>674</Hsp_bit-score>\n              <Hsp_evalue>1e-70</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>490</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>490</Hsp_hit-to>\n              <Hsp_identity>388</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>490</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n        <Hit>\n          <Hit_num>5</Hit_num>\n          <Hit_id>Acidobacterium_capsulatum_ATCC51196</Hit_id>\n          <Hit_def>Acidobacterium capsulatum ATCC 51196</Hit_def>\n          <Hit_accession>NR_074106.1</Hit_accession>\n          <Hit_len>506</Hit_len>\n          <Hit_hsps>\n            <Hsp>\n              <Hsp_num>1</Hsp_num>\n              <Hsp_bit-score>676</Hsp_bit-score>\n              <Hsp_evalue>1e-65</Hsp_evalue>\n              <Hsp_query-from>1</Hsp_query-from>\n              <Hsp_query-to>506</Hsp_query-to>\n              <Hsp_hit-from>1</Hsp_hit-from>\n              <Hsp_hit-to>506</Hsp_hit-to>\n              <Hsp_identity>394</Hsp_identity>\n              <Hsp_gaps>0</Hsp_gaps>\n              <Hsp_align-len>506</Hsp_align-len>\n            </Hsp>\n          </Hit_hsps>\n        </Hit>\n      </Iteration_hits>\n    </Iteration>\n  </BlastOutput_iterations>\n</BlastOutput>\n",
  "pilot_16s_abundance_table.csv": "sample_id,Soil_ASV_A,Soil_ASV_B,note\nRhizosphere_A,128,34,Bacillus-like ASV is more abundant in this toy sample.\nCompost_B,76,93,Both ASVs are detectable in the compost-style toy sample.\nRoot_Nodule_C,21,156,Rhizobium-like ASV is more abundant in this toy sample.\n",
  "pilot_16s_manifest.json": "{\n  \"title\": \"Class-safe soil 16S phylogeny pilot cache\",\n  \"retrieved_date\": \"2026-05-25\",\n  \"reference_count\": 5,\n  \"query_count\": 2,\n  \"cached_hit_table\": \"pilot_16s_cached_hits.csv\",\n  \"cached_blast_xml\": \"pilot_16s_cached_blast.xml\",\n  \"default_mode\": \"Use these cached files; do not require live BLAST or Entrez during class.\",\n  \"references\": [\n    {\n      \"label\": \"Bacillus_subtilis_168\",\n      \"accession\": \"NR_102783.2\",\n      \"source_url\": \"https://www.ncbi.nlm.nih.gov/nuccore/NR_102783.2\"\n    },\n    {\n      \"label\": \"Pseudomonas_fluorescens_CCM2115\",\n      \"accession\": \"NR_115715.1\",\n      \"source_url\": \"https://www.ncbi.nlm.nih.gov/nuccore/NR_115715.1\"\n    },\n    {\n      \"label\": \"Streptomyces_coelicolor_rrnD\",\n      \"accession\": \"Y00411.1\",\n      \"source_url\": \"https://www.ncbi.nlm.nih.gov/nuccore/Y00411.1\"\n    },\n    {\n      \"label\": \"Rhizobium_leguminosarum_IAM12609\",\n      \"accession\": \"D14513.1\",\n      \"source_url\": \"https://www.ncbi.nlm.nih.gov/nuccore/D14513.1\"\n    },\n    {\n      \"label\": \"Acidobacterium_capsulatum_ATCC51196\",\n      \"accession\": \"NR_074106.1\",\n      \"source_url\": \"https://www.ncbi.nlm.nih.gov/nuccore/NR_074106.1\"\n    }\n  ],\n  \"teaching_queries\": [\n    {\n      \"label\": \"Soil_ASV_A\",\n      \"derived_from\": \"NR_102783.2\",\n      \"caution\": \"Synthetic classroom query; not a new environmental isolate.\"\n    },\n    {\n      \"label\": \"Soil_ASV_B\",\n      \"derived_from\": \"D14513.1\",\n      \"caution\": \"Synthetic classroom query; not a new environmental isolate.\"\n    }\n  ]\n}\n"
}

OKABE_ITO = {
    "orange": "#E69F00",
    "sky_blue": "#56B4E9",
    "bluish_green": "#009E73",
    "yellow": "#F0E442",
    "blue": "#0072B2",
    "vermillion": "#D55E00",
    "reddish_purple": "#CC79A7",
    "black": "#222222",
    "gray": "#DDDDDD",
}

BASE_COLORS = {
    "A": OKABE_ITO["sky_blue"],
    "C": OKABE_ITO["bluish_green"],
    "G": OKABE_ITO["orange"],
    "T": OKABE_ITO["reddish_purple"],
    "N": "#BDBDBD",
    "-": "#E6E6E6",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#444444",
    "axes.labelcolor": "#222222",
    "text.color": "#222222",
    "xtick.color": "#222222",
    "ytick.color": "#222222",
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titleweight": "regular",
    "savefig.facecolor": "white",
})
sns.set_theme(style="white")

print("Ready. Dependencies are available and the embedded class cache is loaded in memory.")


## 1. Why cached data?

What can fail during a live class? A public database can be slow, a web API can throttle, or a student can lose time debugging a connection.

So this pilot uses a pre-cached reference set. The biology is still real enough for the lesson: the five reference records are NCBI 16S rRNA sequences from soil-relevant bacteria. The two query reads are clearly marked teaching reads derived from that cache.

**Think:** Why is cached data better for the first teaching run than live BLAST?


In [ ]:
#@title Load cache from GitHub or embedded fallback
def load_cache_file(filename):
    if USE_GITHUB_CACHE and CACHE_BASE_URL.strip():
        url = CACHE_BASE_URL.rstrip("/") + "/" + filename
        try:
            with urllib.request.urlopen(url, timeout=20) as handle:
                text = handle.read().decode("utf-8")
            print(f"Loaded {filename} from GitHub cache.")
            return text
        except Exception as exc:
            print(f"GitHub cache failed for {filename}: {exc}")
            print("Using embedded fallback copy.")
    return EMBEDDED_CACHE[filename]

ref_fasta = load_cache_file("pilot_16s_references.fasta")
query_fasta = load_cache_file("pilot_16s_query_reads.fasta")
metadata_csv = load_cache_file("pilot_16s_metadata.csv")
cached_hits_csv = load_cache_file("pilot_16s_cached_hits.csv")
cached_blast_xml = load_cache_file("pilot_16s_cached_blast.xml")
abundance_csv = load_cache_file("pilot_16s_abundance_table.csv")
manifest_json = load_cache_file("pilot_16s_manifest.json")

references = list(SeqIO.parse(StringIO(ref_fasta), "fasta"))
queries = list(SeqIO.parse(StringIO(query_fasta), "fasta"))
all_records = references + queries
metadata = pd.read_csv(StringIO(metadata_csv))
cached_hits = pd.read_csv(StringIO(cached_hits_csv))
abundance = pd.read_csv(StringIO(abundance_csv))
manifest = json.loads(manifest_json)

def parse_cached_blast_xml(xml_text):
    root = ET.fromstring(xml_text)
    rows = []
    for iteration in root.findall(".//Iteration"):
        query = iteration.findtext("Iteration_query-def")
        for hit in iteration.findall("./Iteration_hits/Hit"):
            hsp = hit.find("./Hit_hsps/Hsp")
            align_len = int(hsp.findtext("Hsp_align-len"))
            identity = int(hsp.findtext("Hsp_identity"))
            rows.append({
                "query_label": query,
                "rank": int(hit.findtext("Hit_num")),
                "reference_label": hit.findtext("Hit_id"),
                "reference_accession": hit.findtext("Hit_accession"),
                "reference_species": hit.findtext("Hit_def"),
                "compared_bases": align_len,
                "differences": align_len - identity,
                "percent_identity_teaching_window": 100 * identity / align_len,
                "bit_score": float(hsp.findtext("Hsp_bit-score")),
                "teaching_e_value": hsp.findtext("Hsp_evalue"),
            })
    return pd.DataFrame(rows)

cached_blast_hits = parse_cached_blast_xml(cached_blast_xml)

print(f"Reference sequences: {len(references)}")
print(f"Query reads: {len(queries)}")
print(f"Cached BLAST-like XML hits: {len(cached_blast_hits)}")
print("Cache retrieved date:", manifest["retrieved_date"])


In [ ]:
#@title Show the teaching reference set
def wrapped_table(df, columns):
    rows = []
    for _, row in df[columns].iterrows():
        cells = "".join(
            f"<td style='padding:8px 10px; border-top:1px solid #e5e5e5; vertical-align:top;'>{row[col]}</td>"
            for col in columns
        )
        rows.append(f"<tr>{cells}</tr>")
    header = "".join(
        f"<th style='text-align:left; padding:7px 10px; border-bottom:1px solid #777;'>{col}</th>"
        for col in columns
    )
    html = f'''
    <div style='font-family: system-ui, Segoe UI, sans-serif; max-width: 980px;'>
      <table style='border-collapse: collapse; table-layout: fixed; width: 100%; font-size: 13px; line-height: 1.35;'>
        <thead><tr>{header}</tr></thead>
        <tbody>{''.join(rows)}</tbody>
      </table>
    </div>
    '''
    display(HTML(html))

display_cols = ["label", "role", "species_or_query", "phylum", "soil_context", "note"]
wrapped_table(metadata, display_cols)


## 2. The class-safe workflow

What should happen when a student presses **Run all**?

The notebook should load known files, make the same comparison for everyone, and produce interpretable figures. Live BLAST and live database searches are useful later, but they are not allowed to be the first-class path because they can fail during class.

The workflow is:

**cached references + cached query reads -> marker window -> alignment view -> distance matrix -> UPGMA/NJ trees -> cautious report**


In [ ]:
#@title Workflow map
steps = [
    ("cache", "cached FASTA + metadata"),
    ("window", "shared 16S marker window"),
    ("align", "aligned position view"),
    ("distance", "pairwise distances"),
    ("tree", "UPGMA + NJ trees"),
    ("claim", "closest-reference report"),
]

fig, ax = plt.subplots(figsize=(11, 2.2))
ax.set_xlim(-0.5, len(steps) - 0.5)
ax.set_ylim(-0.5, 1.05)
ax.axis("off")

for i, (short, label) in enumerate(steps):
    ax.scatter(i, 0.35, s=160, color=OKABE_ITO["sky_blue"], edgecolor="#222222", linewidth=0.8, zorder=3)
    ax.text(i, 0.72, label, ha="center", va="bottom", fontsize=9, wrap=True)
    ax.text(i, 0.35, str(i + 1), ha="center", va="center", fontsize=9, color="white", weight="bold")
    if i < len(steps) - 1:
        ax.plot([i + 0.15, i + 0.85], [0.35, 0.35], color="#999999", linewidth=1.0, zorder=1)

ax.set_title("Class-safe path: no live web service is required during the lesson", loc="left", fontsize=12)
plt.tight_layout()
plt.show()


## 3. Where the references come from

In a real soil microbiome project, a 16S read can be compared against databases such as NCBI BLAST/GenBank, SILVA, RDP, and GTDB.

For this pilot, we already did the retrieval step before class and cached a small reference set. That makes the first run reliable. Later, the class can replace the teaching cache with references selected from live searches.


In [ ]:
#@title Reference database map
database_rows = pd.DataFrame([
    {
        "resource": "NCBI BLAST / GenBank",
        "helps with": "finding close public sequence records and accessions",
        "class-safe use": "cache selected FASTA and source links before class",
    },
    {
        "resource": "SILVA",
        "helps with": "curated rRNA reference alignment and taxonomy context",
        "class-safe use": "download or export selected references before class",
    },
    {
        "resource": "RDP",
        "helps with": "ribosomal RNA taxonomy and classifier-style teaching comparisons",
        "class-safe use": "cache selected reference records or classifier output",
    },
    {
        "resource": "GTDB",
        "helps with": "modern genome-based bacterial taxonomy context",
        "class-safe use": "use as taxonomy background, not as a live dependency",
    },
])
wrapped_table(database_rows, ["resource", "helps with", "class-safe use"])


## 4. Cached closest-hit table

What would students normally wait for from BLAST?

They would wait for a ranked hit table. For class, we cache that idea too. This table is computed from the same teaching marker window and lets students see the species-finding result before they build a tree.

The E-values and bit scores shown below come from the cached BLAST-like teaching XML. They are included so students can learn how ranked-hit evidence is read, but they are not live NCBI BLAST statistics.


In [ ]:
#@title Cached closest-hit table
xml_top = (
    cached_blast_hits[cached_blast_hits["rank"] == 1]
    .set_index("query_label")["reference_label"]
    .to_dict()
)
csv_top = (
    cached_hits[cached_hits["rank"] == 1]
    .set_index("query_label")["reference_label"]
    .to_dict()
)
assert xml_top == csv_top, f"Cached XML and CSV disagree: {xml_top} vs {csv_top}"

hit_view = (
    cached_blast_hits[cached_blast_hits["rank"] <= 3]
    .merge(
        cached_hits[["query_label", "rank", "fraction_different", "interpretation"]],
        on=["query_label", "rank"],
        how="left",
    )
    .copy()
)
hit_view["percent_identity_teaching_window"] = hit_view["percent_identity_teaching_window"].map(lambda x: f"{float(x):.2f}")
hit_view["fraction_different"] = hit_view["fraction_different"].map(lambda x: f"{float(x):.4f}")
hit_view["bit_score"] = hit_view["bit_score"].map(lambda x: f"{float(x):.1f}")
wrapped_table(
    hit_view[
        [
            "query_label",
            "rank",
            "reference_label",
            "reference_accession",
            "percent_identity_teaching_window",
            "bit_score",
            "teaching_e_value",
            "interpretation",
        ]
    ],
    [
        "query_label",
        "rank",
        "reference_label",
        "reference_accession",
        "percent_identity_teaching_window",
        "bit_score",
        "teaching_e_value",
        "interpretation",
    ],
)
print("Cached BLAST-like XML agrees with the cached hit CSV for the top hit of each query.")
print("E-values and bit scores here are cached teaching values, not fresh live-BLAST statistics.")


## 5. From soil sample to species-finding question

A microbiome table can tell us that a read or ASV exists in a sample. It does not automatically tell us what species it is.

The next question is narrower: which cached reference sequence is each query most similar to?

We answer that with sequence comparison first, then we use a tree to report relatedness.


In [ ]:
#@title Tiny microbiome-style count table
fig, ax = plt.subplots(figsize=(8.5, 3.2))
counts = abundance.set_index("sample_id")[["Soil_ASV_A", "Soil_ASV_B"]]
counts.plot(
    kind="bar",
    stacked=True,
    color=[OKABE_ITO["blue"], OKABE_ITO["orange"]],
    edgecolor="white",
    linewidth=0.7,
    ax=ax,
)
ax.set_ylabel("teaching read count")
ax.set_xlabel("")
ax.set_title("Toy soil samples: two ASVs to identify", loc="left", fontsize=12)
ax.legend(frameon=False, title="")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#eeeeee", linewidth=0.8)
plt.xticks(rotation=0, ha="center")
plt.tight_layout()
plt.show()
wrapped_table(abundance, ["sample_id", "Soil_ASV_A", "Soil_ASV_B", "note"])


## 6. Make a comparable 16S marker window

How can we compare sequences fairly? First, we choose the same marker region.

This pilot trims each reference to a shared 16S starting anchor and then uses Biopython's `PairwiseAligner` to make a small star alignment around one reference coordinate. That keeps the class path simple, but still makes the distance step depend on aligned columns rather than raw string positions.

In the later project notebook, this shortcut can be replaced by MAFFT on the team's cleaned reads and selected references.


In [ ]:
#@title Extract the class marker window
def clean_sequence(record):
    return str(record.seq).upper().replace("U", "T").replace(" ", "")

def marker_window(seq, max_bases):
    anchor = "AGAGTTTGATCCTGGCTCAG"
    start = seq.find(anchor)
    if start < 0:
        backup = seq.find("GAGTTTGATCCTGGCTCAG")
        start = max(backup - 1, 0) if backup >= 0 else 0
    return seq[start : start + max_bases]

windows = {}
for record in all_records:
    seq = clean_sequence(record)
    if record.id.startswith("Soil_ASV"):
        windows[record.id] = seq[:MARKER_WINDOW_BASES]
    else:
        windows[record.id] = marker_window(seq, MARKER_WINDOW_BASES)

def global_align_pair(reference, sequence):
    aligner = PairwiseAligner(
        mode="global",
        match_score=2,
        mismatch_score=-1,
        open_gap_score=-5,
        extend_gap_score=-0.5,
    )
    alignment = aligner.align(reference, sequence)[0]
    ref_blocks, seq_blocks = alignment.aligned
    ref_parts = []
    seq_parts = []
    ref_pos = 0
    seq_pos = 0

    for (ref_start, ref_end), (seq_start, seq_end) in zip(ref_blocks, seq_blocks):
        if ref_start > ref_pos:
            ref_parts.append(reference[ref_pos:ref_start])
            seq_parts.append("-" * (ref_start - ref_pos))
        if seq_start > seq_pos:
            ref_parts.append("-" * (seq_start - seq_pos))
            seq_parts.append(sequence[seq_pos:seq_start])

        ref_parts.append(reference[ref_start:ref_end])
        seq_parts.append(sequence[seq_start:seq_end])
        ref_pos = int(ref_end)
        seq_pos = int(seq_end)

    if ref_pos < len(reference):
        ref_parts.append(reference[ref_pos:])
        seq_parts.append("-" * (len(reference) - ref_pos))
    if seq_pos < len(sequence):
        ref_parts.append("-" * (len(sequence) - seq_pos))
        seq_parts.append(sequence[seq_pos:])

    aligned_ref = "".join(ref_parts)
    aligned_seq = "".join(seq_parts)
    if len(aligned_ref) != len(aligned_seq):
        raise ValueError("PairwiseAligner returned unequal reconstructed alignment lengths")
    return aligned_ref, aligned_seq


def star_align_to_reference(windows, reference_label):
    reference = windows[reference_label]
    ref_len = len(reference)
    base_by_label = {}
    insertions_by_label = {}
    max_insertions = {i: 0 for i in range(ref_len + 1)}

    for label, sequence in windows.items():
        if label == reference_label:
            aligned_ref, aligned_seq = reference, sequence
        else:
            aligned_ref, aligned_seq = global_align_pair(reference, sequence)

        bases = ["-"] * ref_len
        insertions = {}
        ref_pos = 0
        for ref_base, seq_base in zip(aligned_ref, aligned_seq):
            if ref_base == "-":
                insertions.setdefault(ref_pos, []).append(seq_base)
            else:
                if ref_pos < ref_len:
                    bases[ref_pos] = seq_base
                ref_pos += 1

        compact_insertions = {pos: "".join(chars) for pos, chars in insertions.items()}
        for pos, inserted in compact_insertions.items():
            max_insertions[pos] = max(max_insertions.get(pos, 0), len(inserted))
        base_by_label[label] = bases
        insertions_by_label[label] = compact_insertions

    aligned = {}
    for label in windows:
        chars = []
        for pos in range(ref_len):
            inserted = insertions_by_label[label].get(pos, "")
            chars.append(inserted.ljust(max_insertions.get(pos, 0), "-"))
            chars.append(base_by_label[label][pos])
        terminal_insert = insertions_by_label[label].get(ref_len, "")
        chars.append(terminal_insert.ljust(max_insertions.get(ref_len, 0), "-"))
        aligned[label] = "".join(chars)
    return aligned

aligned_windows = star_align_to_reference(windows, "Bacillus_subtilis_168")

window_lengths = pd.DataFrame({
    "raw_marker_bases": {label: len(seq) for label, seq in windows.items()},
    "aligned_columns": {label: len(seq) for label, seq in aligned_windows.items()},
})
display(window_lengths)
if window_lengths["raw_marker_bases"].min() < MARKER_WINDOW_BASES:
    print("Some records are shorter than the selected window; distances use the available bases.")


In [ ]:
#@title Visualize a small alignment window
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

def plot_alignment_window(windows, start=0, width=70):
    labels = list(windows)
    end = min(start + width, min(len(seq) for seq in windows.values()))
    bases = ["A", "C", "G", "T", "N", "-"]
    base_to_int = {base: i for i, base in enumerate(bases)}
    matrix = np.array([
        [base_to_int.get(base, base_to_int["N"]) for base in windows[label][start:end]]
        for label in labels
    ])

    fig, ax = plt.subplots(figsize=(11, 4.1))
    cmap = ListedColormap([BASE_COLORS[base] for base in bases])
    ax.imshow(matrix, aspect="auto", interpolation="nearest", cmap=cmap, vmin=0, vmax=len(bases)-1)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_xticks(range(0, end - start, 10))
    ax.set_xticklabels([str(start + x) for x in range(0, end - start, 10)])
    ax.set_xlabel("aligned marker-window column")
    ax.set_title("Aligned teaching window: conserved columns stay quiet; variable columns carry the signal", loc="left", fontsize=12)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    legend_handles = [Patch(facecolor=BASE_COLORS[base], edgecolor="none", label=base) for base in bases]
    ax.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.2),
        ncol=len(bases),
        frameon=False,
        handlelength=1.0,
        columnspacing=1.0,
    )

    for x in range(matrix.shape[1]):
        column = [windows[label][start + x] for label in labels if start + x < len(windows[label])]
        observed = {base for base in column if base in "ACGT"}
        if len(observed) > 1:
            ax.plot([x, x], [-0.48, -0.25], color="#222222", linewidth=0.7, clip_on=False)

    plt.tight_layout()
    plt.show()

plot_alignment_window(aligned_windows, ALIGNMENT_START, ALIGNMENT_WIDTH)


## 7. Compute sequence distances

Once the sequences are comparable, what is the simplest number we can calculate?

For each pair, we count how many aligned positions differ. Dividing by the number of compared positions gives a fraction different. Small values mean the two marker sequences are similar. Larger values mean they are more different.


In [ ]:
#@title Pairwise distances and closest references
def fraction_different(seq_a, seq_b):
    n = min(len(seq_a), len(seq_b))
    a = seq_a[:n]
    b = seq_b[:n]
    compared = 0
    differences = 0
    for left, right in zip(a, b):
        if left in "N-" or right in "N-":
            continue
        compared += 1
        if left != right:
            differences += 1
    return differences / compared if compared else math.nan

labels = list(aligned_windows)
dist = pd.DataFrame(index=labels, columns=labels, dtype=float)
for left in labels:
    for right in labels:
        dist.loc[left, right] = fraction_different(aligned_windows[left], aligned_windows[right])

query_labels = [record.id for record in queries]
reference_labels = [record.id for record in references]
tree_top = {}
for query in query_labels:
    ranked = dist.loc[query, reference_labels].sort_values()
    tree_top[query] = ranked.index[0]
cached_top = (
    cached_hits[cached_hits["rank"] == 1]
    .set_index("query_label")["reference_label"]
    .to_dict()
)
assert tree_top == cached_top, f"Computed closest hits do not match cached table: {tree_top} vs {cached_top}"

closest = (
    cached_hits[cached_hits["rank"] == 1]
    .rename(columns={
        "query_label": "query",
        "reference_label": "closest_reference",
        "fraction_different": "cached_hit_fraction_different",
        "percent_identity_teaching_window": "percent_similarity",
    })
    [["query", "closest_reference", "cached_hit_fraction_different", "percent_similarity"]]
    .copy()
)
closest["tree_distance_to_closest"] = [
    float(dist.loc[row["query"], row["closest_reference"]])
    for _, row in closest.iterrows()
]
display(closest.style.format({
    "cached_hit_fraction_different": "{:.4f}",
    "percent_similarity": "{:.2f}",
    "tree_distance_to_closest": "{:.4f}",
}))
print("Closest-reference labels match the cached hit table; identity percentages come from the cached direct-hit table.")


In [ ]:
#@title Distance matrix heatmap
fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(
    dist,
    cmap="viridis",
    vmin=0,
    vmax=float(np.nanmax(dist.values)),
    square=True,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "fraction of compared positions that differ"},
    ax=ax,
)
ax.set_title("Pairwise 16S marker distances", loc="left", fontsize=12)
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 8. Build distance trees

How does a distance matrix become a tree?

UPGMA repeatedly joins the closest clusters. It is transparent, but it behaves like lineages changed at roughly similar rates.

Neighbor joining also starts with pairwise distances, but it is less tied to that equal-rate assumption. Comparing both methods helps us see that a tree depends on the data and the method.


In [ ]:
#@title Build UPGMA and neighbor-joining trees
def as_distance_matrix(dist_df):
    names = list(dist_df.index)
    lower = []
    for i, name in enumerate(names):
        lower.append([float(dist_df.iloc[i, j]) for j in range(i + 1)])
    return DistanceMatrix(names, lower)

dm = as_distance_matrix(dist)
constructor = DistanceTreeConstructor()
upgma_tree = constructor.upgma(dm)
nj_tree = constructor.nj(dm)
upgma_tree.rooted = True
nj_tree.rooted = False

print("UPGMA tree and neighbor-joining tree built from the same distance matrix.")


In [ ]:
#@title Plot the tree(s)
def plot_tree(tree, title):
    fig, ax = plt.subplots(figsize=(10, 5.4))
    Phylo.draw(
        tree,
        axes=ax,
        do_show=False,
        show_confidence=False,
        label_func=lambda clade: clade.name if clade.name else "",
    )
    ax.set_title(title, loc="left", fontsize=12)
    ax.set_xlabel("sequence-distance-derived branch length")
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.grid(axis="x", color="#eeeeee", linewidth=0.8)
    plt.tight_layout()
    plt.show()

if TREE_METHOD_TO_SHOW == "UPGMA":
    plot_tree(upgma_tree, "UPGMA distance tree")
elif TREE_METHOD_TO_SHOW == "Neighbor joining":
    plot_tree(nj_tree, "Neighbor-joining distance tree")
else:
    plot_tree(upgma_tree, "UPGMA distance tree")
    plot_tree(nj_tree, "Neighbor-joining distance tree")


## 9. Report the result carefully

What exactly did we build?

We built a distance-based gene tree from one 16S marker window. That can support a closest-reference statement. It does not prove exact species identity, and it is not a full species tree.


In [ ]:
#@title Create a student report sentence
row = closest.set_index("query").loc[QUERY_TO_REPORT]
report = (
    f"{QUERY_TO_REPORT} is closest to {row['closest_reference']} in this cached 16S marker comparison "
    f"({row['percent_similarity']:.2f}% similarity across the teaching window). "
    "Because this is one short 16S region, I can report a closest reference, not prove exact species identity or a complete species tree."
)
display(HTML(f'''
<div style='max-width: 900px; border-left: 4px solid {OKABE_ITO["bluish_green"]}; padding: 10px 14px; background: #fafafa; font-size: 15px; line-height: 1.45;'>
  <b>Careful claim:</b><br>{report}
</div>
'''))


## 10. Optional: where IQ-TREE fits

IQ-TREE is useful, but for a different teaching purpose.

UPGMA and neighbor joining are distance methods. They help students see the bridge from sequence differences to tree geometry.

IQ-TREE is a model-based maximum-likelihood tool. Use it after this lesson if you want students to compare a simple distance tree with a modern model-based tree. Do not make it the default class path for the first run.


In [ ]:
#@title Optional advanced note: MAFFT/IQ-TREE path is off by default
RUN_ADVANCED_IQTREE_PATH = False #@param {type:"boolean"}
if RUN_ADVANCED_IQTREE_PATH:
    print("Advanced path:")
    print("1. Align cleaned project reads and references with MAFFT.")
    print("2. Run IQ-TREE with model selection, for example: iqtree2 -s aligned.fasta -m MFP -B 1000 -T AUTO")
    print("3. Compare the maximum-likelihood tree with the UPGMA/NJ teaching trees.")
    print("Keep this optional so the main class run remains low-friction.")
else:
    print("Advanced IQ-TREE path skipped. The class-safe UPGMA/NJ workflow is complete.")


## 11. Replace the teaching cache later

How does this become your real soil microbiome project?

Keep the notebook structure the same, but replace the teaching FASTA and metadata with your team's cleaned 16S reads and selected database references. The critical rule stays the same: prepare the cache before class, push it to GitHub, and let Colab load known files.


In [ ]:
#@title Project replacement schema
project_schema = pd.DataFrame([
    {
        "file": "project_16s_reads.fasta",
        "required columns or fields": "FASTA id, DNA sequence",
        "example": ">TeamA_ASV_001 sample=Rhizosphere_A",
    },
    {
        "file": "project_16s_references.fasta",
        "required columns or fields": "FASTA id, accession, source database",
        "example": ">Bacillus_ref accession=NR_102783.2 source=NCBI",
    },
    {
        "file": "project_16s_metadata.csv",
        "required columns or fields": "label, role, accession, source_database, source_url, date_retrieved, taxonomy, note",
        "example": "TeamA_ASV_001, query, blank, class sample, blank, 2026-05-25, unknown, cleaned ASV",
    },
    {
        "file": "project_16s_abundance_table.csv",
        "required columns or fields": "sample_id plus one column per ASV",
        "example": "Rhizosphere_A, 128, 34, ...",
    },
])
wrapped_table(project_schema, ["file", "required columns or fields", "example"])

print("GitHub cache pattern:")
print('CACHE_BASE_URL = "https://raw.githubusercontent.com/<org>/<repo>/main/soil_16s_class_cache"')
print("Set USE_GITHUB_CACHE=True only after the cache folder is pushed.")


In [ ]:
#@title Export class outputs
output_dir = Path("soil_microbiome_16s_outputs")
output_dir.mkdir(exist_ok=True)
dist.to_csv(output_dir / "soil_16s_distance_matrix.csv")
closest.to_csv(output_dir / "soil_16s_closest_reference_report.csv", index=False)
Phylo.write(upgma_tree, output_dir / "soil_16s_upgma_tree.newick", "newick")
Phylo.write(nj_tree, output_dir / "soil_16s_neighbor_joining_tree.newick", "newick")
metadata.to_csv(output_dir / "soil_16s_metadata_used.csv", index=False)
print("Wrote outputs to:", output_dir.resolve())


## Final Think Prompts

- Which query has the closest reference in the cached set?
- Which references cluster near each other?
- Does a high 16S similarity prove exact species identity?
- What extra evidence would you want before making a stronger species claim?
- How would the workflow change when you replace these cached teaching reads with your team's real project reads?
